# QuantOne — Colab runner (one notebook per model)

Colab variant of the Kaggle template. Results are written **directly to Google Drive**, so a disconnect loses nothing and re-running resumes automatically (content-addressed results, CLAUDE.md rule 2).

**Setup:** Runtime → Change runtime type → **T4 GPU** (cheapest compute units; this study never needs more). Set `MODEL_KEY` below, Run All, approve the Drive-mount prompt.

In [ ]:
# --- Parameters (edit per copy) ---
MODEL_KEY = "qwen25_3b"      # llama32_3b | qwen25_3b | gemma2_2b | phi35_mini | smollm2_17b
REPO_GIT_URL = "https://github.com/kaushiksai29/QuantOne.git"
N_GPU_LAYERS = -1             # -1 = offload all layers; 0 on CPU runtimes
SEEDS = 3
LIMIT = None                  # None = all 500 tasks; small int for smoke tests

In [ ]:
# --- Mount Drive (persistent results) + clone repo ---
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/QuantOne/results'
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
if not os.path.exists('/content/QuantOne'):
    !git clone {REPO_GIT_URL} /content/QuantOne
%cd /content/QuantOne

In [ ]:
# --- Install pinned deps ---
# Prebuilt wheel first; fall back to source build of the SAME pinned version
# if the wheel can't load (some index builds are musl-linked).
WHEEL_INDEX = "cu124" if N_GPU_LAYERS != 0 else "cpu"
!pip install -q -r requirements.txt --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{WHEEL_INDEX}
try:
    import llama_cpp
except Exception as exc:
    print(f"prebuilt wheel unusable ({exc}); building from source ...")
    !pip uninstall -y -q llama-cpp-python
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on" if N_GPU_LAYERS != 0 else ""
    !pip install -q llama-cpp-python==0.3.19 --no-binary llama-cpp-python
    import llama_cpp
print("llama-cpp-python", llama_cpp.__version__)

In [ ]:
# --- Run this model, all quants (GGUFs downloaded/deleted one at a time) ---
limit_arg = f"--limit {LIMIT}" if LIMIT else ""
!python scripts/kaggle_session.py --model {MODEL_KEY} --seeds {SEEDS} \
    {limit_arg} --n-gpu-layers {N_GPU_LAYERS} --results-dir {RESULTS_DIR}

If the manifest above is short, just **Run All again** — results live on Drive, so it resumes where it stopped. When it prints complete counts for this model, you're done: the result JSONs are already in `MyDrive/QuantOne/results/`, alongside any other model run the same way.